In [ ]:
import os
import utils
import random
import numpy as np
import pickle as pkl
from sklearn.metrics import roc_auc_score
from collections import defaultdict

from tqdm import tqdm
from torch.optim import Adam
from sklearn.model_selection import StratifiedKFold

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Configuration

In [ ]:
_DATA = 'diabetes'
_NUM_CONDITION = 10
_SEED = 42
_SHOT = 0
_NUM_EXAMPLE = 5

llm_model = 'gpt-4o'
_API_KEY = ''

# Train/Test Split

In [ ]:
utils.set_seed(_SEED)
df, X_train, X_test, y_train, y_test, target_attr, label_list, is_cat = utils.get_dataset(_DATA, _SEED, _SHOT)
multiclass = True if len(label_list) > 2 else False
y_test = [label_list.index(l) for l in y_test]

X_all = df.drop(target_attr, axis=1)
y_all = np.array([label_list.index(l) for l in df[target_attr]])

In [ ]:
if _DATA in ['communities', 'myocardial']:
    with open(f'data/{_DATA}_selected_column_indices.pkl', 'rb') as f:
        selected_column_indices = pkl.load(f)
    
    X_all = X_all.iloc[:,selected_column_indices]
    X_test = X_test.iloc[:,selected_column_indices]
    
    if X_train is not None:
        X_train = X_train.iloc[:,selected_column_indices]

X_all

In [ ]:
_NUM_GROUP_TOTAL = min(max(X_all.shape[0] // 10, 200), 1000)
_NUM_GROUP_BATCH = 20

print(_NUM_GROUP_TOTAL, _NUM_GROUP_BATCH)

# Data Information

In [ ]:
TASK_DICT = {
    'blood': "Did the person donate blood? Yes or no?",
    'credit-g': "Does this person receive a credit? Yes or no?",
    'diabetes': "Does this patient have diabetes? Yes or no?",
    'heart': "Does the coronary angiography of this patient show a heart disease? Yes or no?",
    'adult': "Does this person earn more than 50000 dollars per year? Yes or no?",
    'bank': "Does this client subscribe to a term deposit? Yes or no?",
    'car': "How would you rate the decision to buy this car? Unacceptable, acceptable, good or very good?",
    'communities': "How high will the rate of violent crimes per 100K population be in this area. Low, medium, or high?",
    'myocardial': "Does the myocardial infarction complications data of this patient show chronic heart failure? Yes or no?"
}

task_desc = TASK_DICT[_DATA]
print(task_desc)

In [ ]:
meta_data_name = f"./data/{_DATA}-metadata.json"
feature_desc = utils.get_feature_desc(X_all, is_cat, meta_data_name)
print(feature_desc)

In [ ]:
label_desc = "[" + ','.join([str(l) for l in label_list]) + "]"
print(label_desc)

# Generate Groups

In [ ]:
random.seed(42)

template_generate_groups = 'templates/generate_groups.txt'
template_ask_for_function = 'templates/ask_for_function.txt'

group_conditions = []
group_titles = []
groups = []

print(f"\n{'Trial':<8} {'# Batches':<15} {'# Groups':<15}")
print("-" * 40)

total_num_group = 0
trial = 0

while total_num_group < _NUM_GROUP_TOTAL:

    random.seed(42 + trial)
    
    ##### Generate Groups #####
    example_indices = random.sample(list(X_all.index), k=_NUM_EXAMPLE)
    examples = ''
    for i, _index in enumerate(example_indices):
        examples += f'Sample {i+1}:\n'
        examples += X_all.iloc[int(_index)].to_string() + '\n\n'

    fill_in_dict = {
        "[NUM_GROUP]": str(_NUM_GROUP_BATCH),
        "[TASK]": task_desc,
        "[FEATURES]": feature_desc,
        "[EXAMPLES]": examples,
        "[LABEL_LIST]": label_desc
    }

    prompt = utils.get_prompt(template_generate_groups, fill_in_dict)
    response = utils.query_gpt(prompt, _API_KEY, max_tokens=1500, temperature=0, model=llm_model)

    try:
        new_group_titles, new_group_conditions = utils.parse_groups(response, "Group")
    except:
        print('Group parsing error')
        continue

    ##### Function #####
    new_groups = []
    for group_index, conditions in enumerate(new_group_conditions):
    
        fill_in_dict = {
            "[FEATURES]": feature_desc,
            "[DESCRIPTION]": new_group_titles[group_index],
            "[CONDITIONS]": " AND ".join(conditions)
        }

        prompt = utils.get_prompt(template_ask_for_function, fill_in_dict)
        response = utils.query_gpt(prompt, _API_KEY, max_tokens=1500, temperature=0, model='gpt-4o-mini')
        function_str = response.split('<start>')[1].split('<end>')[0].strip()
        
        index_list = utils.run_function(function_str, X_all)
        new_groups.append(index_list)

    valid_indices = [j for j in range(len(new_groups)) if len(new_groups[j]) > 1 and isinstance(new_groups[j], list)]
    new_group_conditions = [new_group_conditions[j] for j in valid_indices]
    new_group_titles = [new_group_titles[j] for j in valid_indices]
    new_groups = [new_groups[j] for j in valid_indices]
    
    group_conditions.append(new_group_conditions)
    group_titles.append(new_group_titles)
    groups.append(new_groups)

    total_num_group += len(new_groups)
    trial += 1
    
    print(f"{trial:<8} {len(groups):<15} {total_num_group:<15}")

In [ ]:
group_conditions = [item for sublist in group_conditions for item in sublist][:_NUM_GROUP_TOTAL]
group_titles = [item for sublist in group_titles for item in sublist][:_NUM_GROUP_TOTAL]
groups = [item for sublist in groups for item in sublist][:_NUM_GROUP_TOTAL]
print(len(group_conditions), len(group_titles), len(groups))

In [ ]:
save_dir = f'groups/{llm_model}'
os.makedirs(save_dir, exist_ok=True)

with open(f'{save_dir}/{_DATA}_{_NUM_GROUP_BATCH}_{_NUM_GROUP_TOTAL}_{_NUM_EXAMPLE}_groups.pkl', 'wb') as f:
    pkl.dump([group_conditions, group_titles], f)

with open(f'{save_dir}/{_DATA}_{_NUM_GROUP_BATCH}_{_NUM_GROUP_TOTAL}_{_NUM_EXAMPLE}_indices.pkl', 'wb') as f:
    pkl.dump(groups, f)

In [ ]:
f'{save_dir}/{_DATA}_{_NUM_GROUP_BATCH}_{_NUM_GROUP_TOTAL}_{_NUM_EXAMPLE}_indices.pkl'

# Evaluation

In [ ]:
llm_model = 'gpt-4o'

auroc_dict = defaultdict(list)

### Load Groups ###
save_dir = f'groups/{llm_model}'

with open(f'{save_dir}/{_DATA}_{_NUM_GROUP_BATCH}_{_NUM_GROUP_TOTAL}_{_NUM_EXAMPLE}_indices.pkl', 'rb') as f:
    groups = pkl.load(f)

H = utils.generate_hypergraph(X_all.shape[0], groups)

_NUM_TRIAL = 5
(__NUM_CONDITION, __NUM_EXAMPLE) = (20, 5)

for trial in range(_NUM_TRIAL):
    
    save_dir = f'conditions/{_SHOT}_shot/{trial}_trial/{llm_model}'
    with open(f'{save_dir}/{_DATA}_{__NUM_CONDITION}_{_SEED}_{__NUM_EXAMPLE}_condition2index.pkl', 'rb') as f:
        condition2index = pkl.load(f)

    X = utils.generate_feature(condition2index, X_all.shape[0], len(label_list), __NUM_CONDITION)
    Z = utils.propagate_hypergraph(H, X, alpha=0.85, max_iter=2)[2]
    auroc_X = utils.evaluate(X, X_test.index, y_all, multiclass, len(label_list), __NUM_CONDITION)
    auroc_Z = utils.evaluate(Z, X_test.index, y_all, multiclass, len(label_list), __NUM_CONDITION)
    auroc_dict[f'X {__NUM_CONDITION} {__NUM_EXAMPLE}'].append(auroc_X)
    auroc_dict[f'X + H {__NUM_CONDITION} {__NUM_EXAMPLE}'].append(auroc_Z)

save_dir = f'conditions/{_SHOT}_shot/aggregated/{llm_model}'
with open(f'{save_dir}/{_DATA}_{__NUM_CONDITION}_{_SEED}_{__NUM_EXAMPLE}_condition2index.pkl', 'rb') as f:
    condition2index = pkl.load(f)

X = utils.generate_feature(condition2index, X_all.shape[0], len(label_list), __NUM_CONDITION)
auroc_X = utils.evaluate(X, X_test.index, y_all, multiclass, len(label_list), __NUM_CONDITION)
auroc_Z = utils.evaluate(Z, X_test.index, y_all, multiclass, len(label_list), __NUM_CONDITION)

In [ ]:
print(f'{"Metric":<15} | {"Mean":>8} | {"Std":>8} | {"Min":>8} | {"Max":>8}')
print('-' * 60)

for key in auroc_dict.keys():
    #print(key)
    vals = np.array(auroc_dict[key])
    auroc_mean, auroc_std = np.mean(vals), np.std(vals)
    auroc_min, auroc_max = np.min(vals), np.max(vals)

    # Print formatted statistics
    print(f'{key:<15} | {auroc_mean:8.4f} | {auroc_std:8.4f} | {auroc_min:8.4f} | {auroc_max:8.4f}')
print('-' * 60)